In [14]:
import pandas as pd
from pathlib import Path

# 数据文件夹路径
data_dir = Path("../query_DEPS_DEV_V1/ground_truth_dataset")

# 读取 CSV
df_packageversions = pd.read_csv(data_dir / "Package_info.csv", low_memory=False)
df_pvp = pd.read_csv(data_dir / "Project_Packageversions.csv", low_memory=False)
df_projects = pd.read_csv("../query_DEPS_DEV_V1/filtered_dataset/Project_info_des.csv", low_memory=False)


# 查看前几行
display(df_packageversions.head())
display(df_pvp.head())
display(df_projects.head())


,System,Name,Version,Licenses,Links,Advisories,VersionInfo,Hashes,DependenciesProcessed,DependencyError,UpstreamPublishedAt,Registries,SLSAProvenance,UpstreamIdentifiers,Purl
0,NPM,@ecl/twig-component-carousel,3.11.1,"[\n ""EUPL-1.2""\n]","[\n {\n ""Label"": ""ORIGIN"",\n ""URL"": ""ht...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 29\n}","[\n {\n ""Hash"": ""vH5da6eUZ94i1AcOsrC6VjSV8...",True,False,1.699345e+15,[],NaN,[],NaN
1,NPM,@douganderson444/panzoom-node,1.1.5,[],"[\n {\n ""Label"": ""ORIGIN"",\n ""URL"": ""ht...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 18\n}","[\n {\n ""Hash"": ""bn6jsFfgQaOqxYcxQLdn+w==""...",True,True,1.670271e+15,[],NaN,[],NaN
2,NPM,@douganderson444/panzoom-node,1.1.1,[],"[\n {\n ""Label"": ""ORIGIN"",\n ""URL"": ""ht...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 14\n}","[\n {\n ""Hash"": ""f882urh9+DaLLfg22vF4Dw==""...",True,True,1.654791e+15,[],NaN,[],NaN
3,NPM,@dreamworld/dw-select,3.1.2-fix-double-click-issue.1,"[\n ""ISC""\n]","[\n {\n ""Label"": ""ORIGIN"",\n ""URL"": ""ht...",[],"{\n ""IsRelease"": false,\n ""Ordinal"": 129\n}","[\n {\n ""Hash"": ""1e+/qEffZeAgXRtIOUmPqw==""...",True,False,1.624260e+15,[],NaN,[],NaN
4,NPM,@discue/ui-components,0.13.0,"[\n ""MIT""\n]","[\n {\n ""Label"": ""ORIGIN"",\n ""URL"": ""ht...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 12\n}","[\n {\n ""Hash"": ""9BaLXgrA89SmryO88KCXZg==""...",True,False,1.656518e+15,[],NaN,[],NaN


,System,Name,Version,ProjectType,ProjectName,RelationProvenance,RelationType
0,NPM,@dms/io,0.9.0,GITHUB,dataminingsupply/dms-io,UNVERIFIED_METADATA,SOURCE_REPO_TYPE
1,NPM,@dvo/fc,0.0.4,GITHUB,isacvale/fc,UNVERIFIED_METADATA,SOURCE_REPO_TYPE
2,NPM,@djie/ui,1.0.17,GITHUB,laihaojie/jie,UNVERIFIED_METADATA,SOURCE_REPO_TYPE
3,NPM,@djie/ui,1.0.16,GITHUB,laihaojie/jie,UNVERIFIED_METADATA,SOURCE_REPO_TYPE
4,NPM,@djie/ws,1.0.8,GITHUB,laihaojie/jie,UNVERIFIED_METADATA,SOURCE_REPO_TYPE


,Project_Information,Licenses,Description,Homepage,OSSFuzz
0,"The project ""lberrocal/npm-packages-template"" ...",[],Template for npm package library configured to...,NaN,NaN
1,"The project ""leaflet/leaflet"" on GitHub is a p...","[\n ""non-standard""\n]",🍃 JavaScript library for mobile-friendly inter...,https://leafletjs.com,NaN
2,"The project ""leaflet/leaflet.fullscreen"" on Gi...","[\n ""ISC""\n]",A fullscreen control for Leaflet,http://leaflet.github.io/Leaflet.fullscreen/,NaN
3,"The project ""leaflet/leaflet.markercluster"" is...","[\n ""MIT""\n]",Marker Clustering plugin for Leaflet,NaN,NaN
4,"The project ""leandrowd/react-responsive-carous...","[\n ""MIT""\n]",React.js Responsive Carousel (with Swipe),http://leandrowd.github.io/react-responsive-ca...,NaN


In [21]:
# 全局去掉项目名中的双引号
df_projects["Project_Information"] = df_projects["Project_Information"].str.replace(
    r'\"([^\"]+)\"', r'\1', regex=True
)

# 保存覆盖原文件
df_projects.to_csv("../query_DEPS_DEV_V1/filtered_dataset/Project_info_desc.csv", index=False)


In [15]:
import pandas as pd
import sqlite3


# 2️⃣ 连接（或创建）SQLite 数据库
db_path = "package_query.db"
conn = sqlite3.connect(db_path)

# 3️⃣ 写入数据库，表名为 stockinfo
df_packageversions.to_sql("packageinfo", conn, if_exists="replace", index=False)

print(f"✅ 已将 stockinfo.csv 写入 {db_path}，表名：packageinfo")

# 4️⃣ 关闭连接
conn.close()


✅ 已将 stockinfo.csv 写入 package_query.db，表名：packageinfo


In [22]:
import duckdb
import pandas as pd
from pathlib import Path


# 读取两个 CSV
df_pvp = pd.read_csv(data_dir / "Project_Packageversions.csv", low_memory=False)
df_projects = pd.read_csv("../query_DEPS_DEV_V1/filtered_dataset/Project_info_desc.csv", low_memory=False)

# 建立 DuckDB 连接
db_path = "project_query.db"
con = duckdb.connect(db_path)

# 写入固定表名
con.execute("DROP TABLE IF EXISTS project_packageversion;")
con.execute("CREATE TABLE project_packageversion AS SELECT * FROM df_pvp;")

con.execute("DROP TABLE IF EXISTS project_info;")
con.execute("CREATE TABLE project_info AS SELECT * FROM df_projects;")

print("✅ 已写入 project_packageversion 和 project_info")

# 关闭连接
con.close()


✅ 已写入 project_packageversion 和 project_info


In [17]:
import duckdb

# 连接到已有的 DuckDB 数据库文件
con = duckdb.connect("../query_DEPS_DEV_V1/query_dataset/project_query.db")

# 读取并展示前 5 行
df_test = con.execute("SELECT * FROM project_packageversion LIMIT 5").df()
print(df_test)

# 也可以读取另一张表
df_info = con.execute("SELECT * FROM project_info LIMIT 5").df()
print(df_info)

# 关闭连接
con.close()


  System      Name Version ProjectType              ProjectName  \
0    NPM   @dms/io   0.9.0      GITHUB  dataminingsupply/dms-io   
1    NPM   @dvo/fc   0.0.4      GITHUB              isacvale/fc   
2    NPM  @djie/ui  1.0.17      GITHUB            laihaojie/jie   
3    NPM  @djie/ui  1.0.16      GITHUB            laihaojie/jie   
4    NPM  @djie/ws   1.0.8      GITHUB            laihaojie/jie   

    RelationProvenance      RelationType  
0  UNVERIFIED_METADATA  SOURCE_REPO_TYPE  
1  UNVERIFIED_METADATA  SOURCE_REPO_TYPE  
2  UNVERIFIED_METADATA  SOURCE_REPO_TYPE  
3  UNVERIFIED_METADATA  SOURCE_REPO_TYPE  
4  UNVERIFIED_METADATA  SOURCE_REPO_TYPE  
                                 Project_Information                Licenses  \
0  The project "lberrocal/npm-packages-template" ...                      []   
1  The project "leaflet/leaflet" on GitHub is a p...  [\n  "non-standard"\n]   
2  The project "leaflet/leaflet.fullscreen" on Gi...           [\n  "ISC"\n]   
3  The project "lea

In [20]:
import pandas as pd
from pathlib import Path

# 数据路径
data_dir = Path("../query_DEPS_DEV_V1/ground_truth_dataset")

# 读取 CSV
df_packageversions = pd.read_csv(data_dir / "Package_info.csv", low_memory=False)
df_pvp = pd.read_csv(data_dir / "Project_Packageversions.csv", low_memory=False)
df_projects = pd.read_csv("../query_DEPS_DEV_V1/filtered_dataset/Project_info_des.csv", low_memory=False)

def preview_df(name, df):
    print(f"=== {name} ===")
    print("第一行：")
    print(df.head(1), "\n")
    print("列名和类型：")
    print(df.dtypes, "\n")

# 打印信息
preview_df("Package_info", df_packageversions)
preview_df("Project_Packageversions", df_pvp)
preview_df("Project_info_des", df_projects)


=== Package_info ===
第一行：
  System                          Name Version            Licenses  \
0    NPM  @ecl/twig-component-carousel  3.11.1  [\n  "EUPL-1.2"\n]   

                                               Links Advisories  \
0  [\n  {\n    "Label": "ORIGIN",\n    "URL": "ht...         []   

                                   VersionInfo  \
0  {\n  "IsRelease": true,\n  "Ordinal": 29\n}   

                                              Hashes  DependenciesProcessed  \
0  [\n  {\n    "Hash": "vH5da6eUZ94i1AcOsrC6VjSV8...                   True   

   DependencyError  UpstreamPublishedAt Registries  SLSAProvenance  \
0            False         1.699345e+15         []             NaN   

  UpstreamIdentifiers  Purl  
0                  []   NaN   

列名和类型：
System                    object
Name                      object
Version                   object
Licenses                  object
Links                     object
Advisories                object
VersionInfo               obje

In [9]:
import json
import random
from tqdm import tqdm
from openai import AzureOpenAI
import os
# ====== Azure OpenAI 配置 ======
client = AzureOpenAI(
        api_key=os.getenv("AZURE_API_KEY_o3"),
        api_version=os.getenv("AZURE_API_VERSION_o3", "2023-05-15"),
        azure_endpoint=os.getenv("AZURE_API_BASE_o3")
    )

deployment_name = "gpt-4o-mini"  # 在 Azure Portal 的 Deployments 中查看

# ====== 文件路径 ======
input_path = "../query_DEPS_DEV_V1/filtered_dataset/Project_info.csv"
output_path = "../query_DEPS_DEV_V1/filtered_dataset/Project_info_des.csv"


def generate_project_info(project_type, name, open_issues, stars, forks):
    prompt = (
        f"Given the following project information:\n"
        f"- Type: {project_type}\n"
        f"- Name: {name}\n"
        f"- Open issues count: {open_issues}\n"
        f"- Stars count: {stars}\n"
        f"- Forks count: {forks}\n\n"
        "Write in natural English that describes this project, "
        "and **explicitly include all the above details without omitting or rephrasing any of them**. "
        "The sentence should sound natural and informative, but it must contain all the provided numbers and names exactly as given."
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that writes concise project summaries."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=80
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ Error generating description: {e}")
        return ""

# ====== 主处理流程 ======
df = pd.read_csv(input_path)

project_infos = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    info = generate_project_info(
        row["Type"],
        row["Name"],
        row["OpenIssuesCount"],
        row["StarsCount"],
        row["ForksCount"]
    )
    project_infos.append(info)
    #print(f"[{row['Name']}] → {info}")

df["Project_Information"] = project_infos

df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"✅ Project descriptions saved to: {output_path}")

  0%|          | 1/770 [00:01<16:15,  1.27s/it]

[lberrocal/npm-packages-template] → The project "lberrocal/npm-packages-template" is hosted on GitHub and currently has 0 open issues, 0 stars, and 0 forks.


  0%|          | 2/770 [00:02<15:33,  1.22s/it]

[leaflet/leaflet] → The project "leaflet/leaflet" on GitHub is a popular open-source library that currently has 521 open issues, 38715 stars, and 5782 forks, making it a widely recognized tool in the developer community.


  0%|          | 3/770 [00:03<14:33,  1.14s/it]

...

[tmpvar/jsdom] → The project tmpvar/jsdom is hosted on GitHub, where it currently has 479 open issues, 19,356 stars, and 1,668 forks, making it a popular choice among developers looking to work with a JavaScript environment that simulates a web browser.


100%|█████████▉| 769/770 [13:33<00:01,  1.23s/it]

[toilal/ng-pickadate] → The project named toilal/ng-pickadate is hosted on GitHub and currently has 15 open issues, 71 stars, and 34 forks.


100%|██████████| 770/770 [13:34<00:00,  1.06s/it]

[wizards-lab/routing] → The project named wizards-lab/routing is hosted on GITLAB and currently has an open issues count of 0, a stars count of 0, and a forks count of 0.
✅ Project descriptions saved to: ../query_DEPS_DEV_V1/filtered_dataset/Project_info_des.csv


In [12]:
cols_to_drop = ["Type", "Name", "OpenIssuesCount", "StarsCount", "ForksCount"]
#df.drop(columns=cols_to_drop, inplace=True)

# 把 Project_Information 移到最前面
cols = ["Project_Information"] + [c for c in df.columns if c != "Project_Information"]
df = df[cols]
df
df.to_csv(output_path, index=False, encoding="utf-8-sig")

In [5]:
import pandas as pd
import json

# 读取 PACKAGEVERSIONS.csv
# df_packageversions = pd.read_csv("PACKAGEVERSIONS.csv")

# 找出 Name 出现次数大于 1 的记录
duplicate_names = df_packageversions["Name"].value_counts()
duplicate_names = duplicate_names[duplicate_names > 1]

# 保留这些重复 Name 对应的完整行
duplicate_rows = df_packageversions[df_packageversions["Name"].isin(duplicate_names.index)].copy()

# 解析 VersionInfo 里的 Ordinal
def extract_ordinal(version_info):
    try:
        data = json.loads(version_info)
        return float(data.get("Ordinal", None))
    except Exception:
        return None

duplicate_rows["Ordinal"] = duplicate_rows["VersionInfo"].apply(extract_ordinal)

# 按 Name 和 Ordinal 排序（Ordinal 降序方便看最新版本）
duplicate_rows = duplicate_rows.sort_values(["Name", "Ordinal"], ascending=[True, False])

print(f"共有 {len(duplicate_names)} 个 Name 是重复的")
duplicate_rows.head(20)


共有 16222 个 Name 是重复的


,System,Name,Version,Licenses,Links,Advisories,VersionInfo,Hashes,DependenciesProcessed,DependencyError,UpstreamPublishedAt,Registries,SLSAProvenance,UpstreamIdentifiers,Purl,Ordinal
499904,NPM,@discordx/music,6.0.2,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 213\n}","[\n {\n ""Hash"": ""q55DKR7CDGzcT1ISX0dp3g==""...",True,False,1.687719e+15,[],NaN,[],NaN,213.0
499905,NPM,@discordx/music,6.0.2,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 213\n}","[\n {\n ""Hash"": ""q55DKR7CDGzcT1ISX0dp3g==""...",True,False,1.687719e+15,[],NaN,[],NaN,213.0
271029,NPM,@discordx/music,6.0.1,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 212\n}","[\n {\n ""Hash"": ""jvZnCflfDKcWhTCjfC0IhQ==""...",True,False,1.687715e+15,[],NaN,[],NaN,212.0
271030,NPM,@discordx/music,6.0.1,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 212\n}","[\n {\n ""Hash"": ""jvZnCflfDKcWhTCjfC0IhQ==""...",True,False,1.687715e+15,[],NaN,[],NaN,212.0
101210,NPM,@discordx/music,6.0.0,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 211\n}","[\n {\n ""Hash"": ""dd+OLx7rDYJuYX38ftB+FQ==""...",True,False,1.687713e+15,[],NaN,[],NaN,211.0
101211,NPM,@discordx/music,6.0.0,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 211\n}","[\n {\n ""Hash"": ""dd+OLx7rDYJuYX38ftB+FQ==""...",True,False,1.687713e+15,[],NaN,[],NaN,211.0
475992,NPM,@discordx/music,6.0.0-dev.1676225191.4821b1ea,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": false,\n ""Ordinal"": 210\n}","[\n {\n ""Hash"": ""+Ud8xf7R9rIRcDzfjHa57g==""...",True,False,1.676225e+15,[],NaN,[],NaN,210.0
475993,NPM,@discordx/music,6.0.0-dev.1676225191.4821b1ea,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": false,\n ""Ordinal"": 210\n}","[\n {\n ""Hash"": ""+Ud8xf7R9rIRcDzfjHa57g==""...",True,False,1.676225e+15,[],NaN,[],NaN,210.0
221649,NPM,@discordx/music,5.0.2,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 209\n}","[\n {\n ""Hash"": ""5+xqsyRxvCQcX+3G5s7Hmg==""...",True,False,1.679481e+15,[],NaN,[],NaN,209.0
221650,NPM,@discordx/music,5.0.2,"[\n ""Apache-2.0""\n]","[\n {\n ""Label"": ""HOMEPAGE"",\n ""URL"": ""...",[],"{\n ""IsRelease"": true,\n ""Ordinal"": 209\n}","[\n {\n ""Hash"": ""5+xqsyRxvCQcX+3G5s7Hmg==""...",True,False,1.679481e+15,[],NaN,[],NaN,209.0
